Modules:

In [1]:
import pandas as pd
import time
from google import genai
from google.genai import types
import json
import os
import sys
from tqdm import tqdm
from dotenv import load_dotenv

Config:

In [25]:
# --- CONFIGURATION ---
# Load the API key from your specific .env file
load_dotenv("API_keys.env")
API_KEY = os.getenv("GEMINI_API_KEY_KOBE")
if API_KEY: print("API_KEY loaded succesfully")

# Prompt Technique Toggles
EXAMPLES_ENABLED = False    # 3-shot examples
REASONING_ENABLED = True   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
ROLE_NR = 1                   # Choose the system prompt (role): default = 1
ROLE = {1: "an expert in ESG investment", 
         2: "an expert in sentiment analysis", 
         3: "a professor of linguistics",
         4: "an academic expert in ESG (Environmental, Social, and Governance) analysis"}[ROLE_NR]

# --- FILES & MODEL ---
MODEL = "Gem2.5f"
MODEL_ID = {"Gem2.5f": "gemini-2.5-flash"}[MODEL]
OUTPUT_FILE = f"PoC sentiment_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES_ENABLED else ""}{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.csv"
INPUT_FILE = "PoC data.csv" if not os.path.exists(OUTPUT_FILE) else OUTPUT_FILE     # Continue with the previous output if output has already been created for this configuration
EXAMPLES_FILE = f"sentiment_examples{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.txt"

# Script Parameters
TEMPERATURE = 0                       # Strictly deterministic
MAX_POSTS = 2                        # "None" for full analysis
POST_START = 17                       # Index of the post where analysis should start (0 is first post)
CHECKPOINT_INTERVAL = 5              # Save after x posts
SLEEP_TIME = 14                        # Break (in seconds) between API calls
RETRIES = 3                             # Amount of retries for 500 / 503 errors


API_KEY loaded succesfully


# SA algorithm:

In [ ]:
client = genai.Client(api_key=API_KEY)

def load_examples(filepath):
    """Loads few-shot examples from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Examples file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
        
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def build_prompt(post_text, examples):
    """Dynamically builds the prompt based on configuration."""
    
    # Construct dynamic schema description for the prompt text
    schema_desc = ['"sentiment": "1 / 0 / -1 / N/A"']
    if REASONING_ENABLED: schema_desc.append('"reasoning": "string"')
    if CONFIDENCE_ENABLED: schema_desc.append('"confidence": float')

    prompt = f"""
You are {ROLE}.
Your task is to analyze LinkedIn posts and evaluate the overall affective orientation and sentiment. 
This evaluation must be conducted at the document level, meaning you must assess the holistic message and strategic positioning of the entire post rather than focusing on isolated sentences or fragmented emotions.

Instructions:
1. Determine whether the text contains any subjective evaluation, opinion, or emotion. 
2. If the text is purely objective or informational, it must be classified as N/A. 
3. If the text contains subjective elements, you must classify its overall polarity on a discrete 3-point scale: Positive (value of 1), Neutral (value of 0), or Negative (value of -1)
{ "4. Provide a detailed reasoning for every active sentiment classification (including posts classified as 'N/A')." if REASONING_ENABLED else "" }
{ "5. Provide a confidence score (0.0 to 1.0) for every classification (including posts classified as 'N/A'), representing your level of certainty." if CONFIDENCE_ENABLED else "" }


Sentiment scale:
1. POSITIVE (1): Select this category if the overall tone of the post expresses a favorable opinion, optimism, praise, or satisfaction. 
In the context of corporate social media, this includes posts that use evaluative language to highlight corporate achievements, successful sustainability initiatives, or enthusiastic endorsements of events and policies.
2. NEGATIVE (-1): Select this category if the overall tone of the post expresses an unfavorable opinion, criticism, dissatisfaction, disappointment, or a pessimistic outlook. 
This includes posts that use evaluative language to express concern over societal issues, failures to meet targets, or frustration with current regulations.
3. NEUTRAL (0): Select this category if the post is unequivocally subjective and contains evaluative language, but the overall polarity does not lean distinctly towards positive or negative. 
This category strictly applies to two scenarios:
Mixed Sentiment: The post contains an active and relatively equal mixture of both strong positive and strong negative sentiments, balancing each other out.
Explicitly Neutral Sentiment: The post expresses a mild, mediocre, or deliberately middle-of-the-road opinion.
4. N/A: Select this category if the post lacks any form of subjective opinion, emotion, or evaluative language. 
This category is reserved for purely factual, informational, or functional texts that do not belong on the sentiment polarity continuum.

Important Annotation Rules:
Do not confuse 'Neutral' with 'N/A'. 'Neutral' implies the presence of an opinion (mixed or moderate), whereas 'N/A' implies the complete absence of an opinion.

{ f"Examples:\n{examples}" if EXAMPLES_ENABLED else "" }

Strict Output Format (JSON):
{{
    {", ".join(schema_desc)}
}}

Input (LinkedIn Post):
{post_text}
"""
    return prompt

def get_response_schema():
    """Defines a strict JSON schema for a single sentiment output."""
    properties = {
        "sentiment": {
            "type": "STRING", 
            "description": "The sentiment of the post: '1' (positive), '0' (neutral), '-1' (negative), or 'N/A' (not applicable)."
        }
    }

    # Add optional properties based on config toggles
    if REASONING_ENABLED:
        properties["reasoning"] = {
            "type": "STRING", 
            "description": "Short explanation of why this sentiment was chosen."
        }
    if CONFIDENCE_ENABLED:
        properties["confidence"] = {
            "type": "NUMBER", 
            "description": "Confidence score from 0.0 to 1.0."
        }
    
    return types.Schema(
        type="OBJECT",
        properties=properties,
        required=list(properties.keys())
    )

def classify_post(post_text, examples, retries):
    """Sends prompt to Gemini using a strict Response Schema and handles 503 errors."""
    prompt = build_prompt(post_text, examples)
    
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=get_response_schema(), # This is the magic fix
                )
            )
            # The schema ensures the text is valid JSON
            return json.loads(response.text)
            
        except Exception as e:
            # Check for 503 (Service Unavailable) or 500 (Internal Error)
            if "503" in str(e) or "500" in str(e):
                print(f"\n[Attempt {attempt + 1}] Server busy (503/500). Retrying in {5 * (attempt + 1)}s...")
                time.sleep(5 * (attempt + 1))
                continue
            
            # For other errors (like auth or quota), return immediately
            return {"error": str(e)}
            
    return {"error": "Maximum retries reached. Google service is currently unavailable."}

def main():
    # 1. Verification of required files
    if EXAMPLES_ENABLED:
        try:
            examples = load_examples(EXAMPLES_FILE)
        except FileNotFoundError:
            sys.exit(1)
    else:
        examples = ""
        
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    # 2. Load Data
    df = pd.read_csv(INPUT_FILE)
    print(INPUT_FILE, "opened succesfully")
    # Expected columns: Company, Date, Link, Text
    text_col = "Text" 

    # 3. Initialize Columns
    # Create the necessary columns based on toggles
    target_cols = ["sentiment"]
    if REASONING_ENABLED: target_cols.append("sentiment_reasoning")
    if CONFIDENCE_ENABLED: target_cols.append("sentiment_confidence")

    for col in target_cols:
        if col not in df.columns: df[col] = None

    # Calculate range
    end_index = len(df)
    if MAX_POSTS is not None:
        end_index = min(POST_START + MAX_POSTS, len(df))

    print(f"Processing posts from index {POST_START} to {end_index}...")

    # 4. Processing Loop
    for index in tqdm(range(POST_START, end_index), desc="Classifying"):
        # Skip if already successfully processed
        if pd.notna(df.at[index, 'sentiment']) and "ERROR" not in str(df.at[index, 'sentiment']):
            continue

        post_content = df.at[index, text_col]
        result = classify_post(post_content, examples, RETRIES)

        if "error" in result:
            df.at[index, 'sentiment'] = "ERROR: " + result["error"]
        else:
            # Map the flat JSON response to the DataFrame
            df.at[index, "sentiment"] = result.get("sentiment", "N/A")

            if REASONING_ENABLED:
                df.at[index, "sentiment_reasoning"] = result.get("reasoning", "")

            if CONFIDENCE_ENABLED:
                df.at[index, "sentiment_confidence"] = result.get("confidence", 0.0)

        # Checkpoint
        if (index + 1) % CHECKPOINT_INTERVAL == 0:
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        
        time.sleep(SLEEP_TIME)

    # Final Save
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

PoC sentiment_Gem2.5f_role1_reasoning.csv opened succesfully
Processing posts from index 17 to 19...


Classifying:   0%|          | 0/2 [00:00<?, ?it/s]


[Attempt 1] Server busy (503/500). Retrying in 5s...


Classifying: 100%|██████████| 2/2 [00:44<00:00, 22.40s/it]


Processing complete! Results saved to PoC sentiment_Gem2.5f_role1_reasoning.csv
